<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 1 - Ranking Signal Analysis**


**Why this one:** I want to understand what actually separates pages that succeed from those that don't. The starter notebooks showed surprising findings - like search volume having zero correlation with actual traffic (0.001) and word count not mattering for decline. This tells me common assumptions might be wrong. I want to systematically test which signals (position, CTR, engagement, content age, etc.) actually relate to a page's performance. This matters because if we know what signals matter, we can give content teams specific, evidence-backed recommendations instead of generic advice.



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**What decision does this improve?**

Deciding which content factors are worth investing in or prioritizing for improvement. Should teams focus on improving CTR, increasing engagement, targeting better positions, or something else?

**Who acts on it?**

Content strategists, SEO teams, and editors who decide where to invest their time and resources.

**What does a wrong call cost?**

- False positive (saying a signal matters when it doesn't): Wasted time optimizing things that don't move the needle or actually help

- False negative (missing a signal that matters): Missing opportunities to improve pages that could perform better

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [27]:
# === SETUP ===

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [28]:
# === Load and analyze data ===

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Quick Data Snapshot ===\n")
print(f"Total pages: {len(df):,}")
print(f"Declining pages: {df[df['trend_direction'] == 'down'].shape[0]:,} ({df[df['trend_direction'] == 'down'].shape[0]/len(df)*100:.1f}%)\n")

# Signal 1: Search volume vs actual traffic (from Discovery A)
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Signal 1: Search volume vs actual traffic correlation: {corr:.3f}")
print("→ Near zero! This challenges the assumption that high search volume = more traffic.\n")

# Signal 2: CTR by position (from Discovery B)
ctr_by_pos = df.groupby('position_tier')['ctr'].mean().sort_values(ascending=False)
print("Signal 2: CTR varies dramatically by position:")
print(ctr_by_pos.round(3))
print("→ Position appears strongly related to CTR (though interpretation requires caution).\n")

# Signal 3: Declining vs growing word count (from Discovery C)
wc_by_trend = df.groupby("trend_direction")["word_count"].median()
print("Signal 3: Median word count by trend:")
for direction, count in wc_by_trend.items():
    print(f"  {direction}: {count:.0f}")
print("→ Declining and growing pages have similar word count. Length isn't the lever.\n")

# BONUS: My own discovery - Engagement doesn't predict more impressions!
df['engagement_group'] = pd.cut(
    df['engagement_rate'],
    bins=[0, 1, 5, 10, 100],
    labels=['Very Low (<1%)', 'Low (1-5%)', 'Medium (5-10%)', 'High (>10%)']
)
engagement_result = df.groupby('engagement_group', observed=False)['impressions_90d'].median()
print("Signal 4 (My discovery): Median impressions by engagement rate:")
print(engagement_result.round(0))
print("→ Surprising: Higher engagement pages had lower median impressions in this data.\n")

print("These patterns show clear signals worth investigating further.")

=== Quick Data Snapshot ===

Total pages: 30,000
Declining pages: 16,262 (54.2%)

Signal 1: Search volume vs actual traffic correlation: 0.001
→ Near zero! This challenges the assumption that high search volume = more traffic.

Signal 2: CTR varies dramatically by position:
position_tier
top_3       1.484
page_1      0.652
striking    0.323
page_3_5    0.222
deep        0.150
Name: ctr, dtype: float64
→ Position appears strongly related to CTR (though interpretation requires caution).

Signal 3: Median word count by trend:
  down: 2909
  flat: 2698
  new: 2239
  stable: 2912
  up: 2848
→ Declining and growing pages have similar word count. Length isn't the lever.

Signal 4 (My discovery): Median impressions by engagement rate:
engagement_group
Very Low (<1%)    11579.0
Low (1-5%)         9323.0
Medium (5-10%)     4767.0
High (>10%)        1130.0
Name: impressions_90d, dtype: float64
→ Surprising: Higher engagement pages had lower median impressions in this data.

These patterns show cl

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I CAN claim (observed, measured, directional):**

- I observed a relationship between position and CTR. Pages in top positions tend to have higher CTR on average.

- I observed an inverse relationship between engagement rate and impressions in this dataset.

- Search volume showed near-zero correlation with actual impressions in this dataset.

- Declining and growing pages had similar median word counts.

- Content age appeared related to impressions, with older pages performing better than very new ones in our sample.

 These patterns suggest the data holds unexpected relationships worth deeper investigation.


**What I CANNOT claim (not proven by this data):**

- High engagement causes fewer impressions(correlation only)

- Old content is always better than new content (sample limitations)

- Position causes higher CTR(correlation ≠ causation)

- Changing content will fix decline (requires experiment)

- This is how Google's algorithm works (we can't know internal logic)

- Any causal claims - this is observational analysis


**What my work will provide:**

Evidence-based recommendations for what content teams should test and prioritize, not guarantees of what will work.

The data revealed unexpected patterns. Contrary to common assumptions, pages with higher engagement had lower median impressions, and older pages appeared to perform better than very new pages. These counter-intuitive findings highlight the value of systematic signal analysis over relying on intuition.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.